# Insurance Claim Processing Agent

## Project Overview

Insurance claim processing involves multiple steps such as verifying supporting documents, checking policy eligibility, identifying potential fraud, and deciding whether a claim can be approved or requires further investigation.

In this project, we will build an **AI-powered Insurance Claim Processing Agent** using **LangGraph** and **OpenAI GPT-4o-mini**.

The agent will accept claim details and supporting document information, perform multiple checks, generate a claim summary, and determine the appropriate next action:

- **Auto Approve** — when the claim is complete, eligible, and has no significant fraud indicators.
- **Reject** — when required documents are missing or the policy is not eligible.
- **Human Review** — when the claim is high-risk, suspicious, high-value, or requires human judgment.

## Key Capabilities

The LangGraph workflow will include the following agents/nodes:

1. **Document Verification Agent**  
   Verifies whether all required claim documents are available and valid.

2. **Eligibility Check Agent**  
   Checks whether the policy is active and whether the claim is covered under the policy.

3. **Fraud Detection Agent**  
   Identifies potential fraud indicators such as unusual claim amounts or suspicious claim patterns.

4. **Claim Summary Agent**  
   Generates a concise summary of the claim and the results of the different checks.

5. **Human Approval Agent**  
   Routes uncertain or high-risk claims to a human for final review and approval.

## AgenticAI Workflow

The project will demonstrate an agentic workflow using **LangGraph**, including:

- Parallel execution of document, eligibility, and fraud checks
- Conditional routing based on agent results
- Human-in-the-Loop for high-risk or uncertain claims
- Structured state management
- LLM-powered reasoning and summarization

## Technology Stack

- **Python**
- **LangGraph**
- **LangChain**
- **OpenAI GPT-4o-mini**
- **Pydantic**
- **Streamlit**
- **Jupyter Notebook / VS Code**

## Expected Outcome

By the end of this project, we will have a working Insurance Claim Processing Agent that can process different claim scenarios and automatically route them to the appropriate outcome.

The final application will be tested using multiple scenarios, including:

1. Complete claim with valid documents → **Auto Approve**
2. Missing required documents → **Reject**
3. Suspicious claim amount → **Human Review**
4. Expired policy → **Reject**
5. High-value claim with valid documents → **Human Review**

The completed solution will also be converted into a **Streamlit application** suitable for demonstration and deployment.

## Project Workflow and Architecture

The Insurance Claim Processing Agent will use **LangGraph** to orchestrate multiple specialized agents.

The workflow is designed so that the three initial assessment agents can work **in parallel**, reducing unnecessary sequential processing.

### High-Level Workflow

**Claim Input**  
↓  
**Initialize Claim State**  
↓  
**Parallel Processing**
- Document Verification Agent
- Eligibility Check Agent
- Fraud Detection Agent
↓  
**Combine Assessment Results**  
↓  
**Decision & Routing**
- Missing documents → **Reject**
- Policy not eligible → **Reject**
- High-risk / suspicious / high-value → **Human Review**
- All checks passed → **Auto Approve**
↓  
**Claim Summary Agent**  
↓  
**Final Decision / Human Approval**  
↓  
**Final Claim Result**

### LangGraph Architecture

The graph will contain the following major nodes:

#### 1. Document Verification Agent

Checks whether all required documents have been submitted.

Example documents:

- Claim Form
- Policy Document
- Identity Proof
- Repair Estimate
- Medical/Incident Documents, where applicable

Output:

- Documents complete or incomplete
- Missing documents
- Verification status

---

#### 2. Eligibility Check Agent

Checks whether the claim is eligible under the policy.

The agent will evaluate information such as:

- Policy status
- Policy start and expiry dates
- Claim date
- Coverage type
- Claim amount
- Policy coverage limits

Output:

- Eligible / Not Eligible
- Reason for the decision

---

#### 3. Fraud Detection Agent

Analyzes the claim for potential fraud indicators.

Examples include:

- Unusually high claim amount
- Suspicious claim patterns
- Inconsistent claim information
- Multiple risk indicators
- Other predefined business rules

Output:

- Fraud risk level
- Risk indicators
- Fraud assessment

---

### Parallel Execution

The following three agents will execute independently:

```text
                 ┌── Document Verification ──┐
                 │                            │
Claim Input ─────┼── Eligibility Check ──────┼──→ Decision Router
                 │                            │
                 └── Fraud Detection ────────┘

## 1. Project Setup

Load the required environment variables and initialize the OpenAI LLM connection.

The OpenAI API key will be stored in the `.env` file rather than directly in the notebook. This keeps sensitive credentials separate from the project code and helps prevent accidental exposure when the project is uploaded to GitHub.

We will use **GPT-4o-mini** as the LLM for this project with a temperature of 0 to provide more consistent outputs.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Remove SSL_CERT_FILE if it points to an invalid certificate location
os.environ.pop("SSL_CERT_FILE", None)

# Load environment variables from .env
load_dotenv()

# Retrieve the OpenAI API key
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found. Please check your .env file.")

# Initialize the OpenAI LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Project setup completed successfully.")
print("LLM initialized:", llm.model_name)

Project setup completed successfully.
LLM initialized: gpt-4o-mini


## 2. Test the LLM Connection

Before building the LangGraph workflow, verify that the configured LLM can successfully process a prompt and return a response.

This confirms that the API key, OpenAI connection, LangChain integration, and selected model are all working correctly.

In [ ]:
response = llm.invoke("In one sentence, explain what an insurance claim is.")

print("LLM Response:")
print(response.content)

LLM Response:
An insurance claim is a formal request made by a policyholder to an insurance company for compensation or coverage for a loss or damage covered under their insurance policy.


## 3. Define the Claim Data Model

Before creating the agents, we need to define the information that our Insurance Claim Processing Agent will work with.

We will use **Pydantic** to create a structured claim model.

The model will contain the basic information required for processing an insurance claim, including:

- Claim ID
- Policy ID
- Claimant name
- Policy status
- Policy expiry date
- Claim date
- Claim type
- Claim amount
- Required documents
- Submitted documents

Using a structured data model helps ensure that claim information is consistent, validated, and easier for the LangGraph agents to process.

In [ ]:
from datetime import date
from pydantic import BaseModel, Field

class InsuranceClaim(BaseModel):
    claim_id: str = Field(..., description="Unique claim identifier")
    policy_id: str = Field(..., description="Insurance policy identifier")
    claimant_name: str = Field(..., description="Name of the claimant")
    policy_status: str = Field(..., description="Current policy status")
    policy_expiry_date: date = Field(..., description="Policy expiry date")
    claim_date: date = Field(..., description="Date on which the claim was made")
    claim_type: str = Field(..., description="Type of insurance claim")
    claim_amount: float = Field(..., gt=0, description="Claim amount")
    required_documents: list[str] = Field(
        default_factory=list,
        description="Documents required for the claim"
    )
    submitted_documents: list[str] = Field(
        default_factory=list,
        description="Documents submitted by the claimant"
    )

print("InsuranceClaim model created successfully.")

InsuranceClaim model created successfully.


## 4. Create a Sample Insurance Claim

Create a realistic sample claim that will be used during the initial development and testing of the workflow.

This claim represents a straightforward case:

- The policy is active.
- The policy has not expired.
- The claim is within a reasonable amount.
- All required documents have been submitted.

The expected final outcome for this scenario will be **Auto Approve** after all agents complete their assessments.

In [8]:
sample_claim = InsuranceClaim(
    claim_id="CLM-1001",
    policy_id="POL-5001",
    claimant_name="John Smith",
    policy_status="Active",
    policy_expiry_date=date(2027, 12, 31),
    claim_date=date(2026, 9, 15),
    claim_type="Vehicle Damage",
    claim_amount=8500.00,
    required_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ],
    submitted_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ]
)

print(sample_claim)

claim_id='CLM-1001' policy_id='POL-5001' claimant_name='John Smith' policy_status='Active' policy_expiry_date=datetime.date(2027, 12, 31) claim_date=datetime.date(2026, 9, 15) claim_type='Vehicle Damage' claim_amount=8500.0 required_documents=['Claim Form', 'Policy Document', 'Identity Proof', 'Repair Estimate'] submitted_documents=['Claim Form', 'Policy Document', 'Identity Proof', 'Repair Estimate']


## 5. Define the LangGraph State

LangGraph uses a shared **state** to pass information between nodes in the workflow.

Our state will contain:

- Original claim information
- Document verification results
- Eligibility assessment
- Fraud assessment
- Claim summary
- Final decision
- Human review status

The three initial assessment agents will independently update their respective sections of the state.

This shared state will later allow LangGraph to execute the Document Verification, Eligibility Check, and Fraud Detection agents in parallel and then combine their results for decision routing.

In [9]:
from typing import TypedDict

class ClaimProcessingState(TypedDict, total=False):
    claim: InsuranceClaim

    document_status: str
    missing_documents: list[str]

    eligibility_status: str
    eligibility_reason: str

    fraud_risk_level: str
    fraud_indicators: list[str]

    claim_summary: str

    final_decision: str
    human_review_required: bool
    human_review_reason: str

print("LangGraph state model created successfully.")

LangGraph state model created successfully.


## 6. Initialize the Claim Processing State

Initialize the LangGraph state using the sample insurance claim created earlier.

At this stage, only the original claim information is populated.

The assessment fields will be populated later by the Document Verification, Eligibility Check, and Fraud Detection agents.

In [10]:
initial_state: ClaimProcessingState = {
    "claim": sample_claim,
    "document_status": "",
    "missing_documents": [],
    "eligibility_status": "",
    "eligibility_reason": "",
    "fraud_risk_level": "",
    "fraud_indicators": [],
    "claim_summary": "",
    "final_decision": "",
    "human_review_required": False,
    "human_review_reason": ""
}

print("Initial claim state created successfully.")
print("\nClaim ID:", initial_state["claim"].claim_id)
print("Policy ID:", initial_state["claim"].policy_id)
print("Claim Amount:", initial_state["claim"].claim_amount)

Initial claim state created successfully.

Claim ID: CLM-1001
Policy ID: POL-5001
Claim Amount: 8500.0


## 7. Document Verification Agent

The Document Verification Agent checks whether all documents required for processing the claim have been submitted.

The agent will:

1. Read the required documents from the claim.
2. Compare them with the submitted documents.
3. Identify any missing documents.
4. Update the shared LangGraph state.
5. Return a clear verification status.

This node uses deterministic business logic rather than an LLM because document presence can be reliably verified through direct comparison.

Possible outcomes:

- **Complete** → All required documents are available.
- **Incomplete** → One or more required documents are missing.

In [ ]:
def document_verification_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    claim = state["claim"]

    required_documents = {
        document.strip().lower()
        for document in claim.required_documents
    }

    submitted_documents = {
        document.strip().lower()
        for document in claim.submitted_documents
    }

    missing_documents = sorted(required_documents - submitted_documents)

    if missing_documents:
        document_status = "Incomplete"
    else:
        document_status = "Complete"

    return {
        "document_status": document_status,
        "missing_documents": missing_documents
    }

document_result = document_verification_agent(initial_state)

print("Document Verification Result")
print("----------------------------")
print("Status:", document_result["document_status"])
print("Missing Documents:", document_result["missing_documents"])

Document Verification Result
----------------------------
Status: Complete
Missing Documents: []


## 8. Create the LangGraph Workflow

Convert the Document Verification Agent into a LangGraph node.

For now, the graph contains only one node:

**START → Document Verification Agent → END**

As we build the project, additional nodes will be added to this graph, including:

- Eligibility Check Agent
- Fraud Detection Agent
- Decision Router
- Claim Summary Agent
- Human Approval Agent

Eventually, the initial three assessment agents will execute in parallel.

In [14]:
from langgraph.graph import StateGraph, START, END

# Create the LangGraph workflow using our shared state
workflow = StateGraph(ClaimProcessingState)

# Add the Document Verification Agent as a node
workflow.add_node("document_verification", document_verification_agent)

# Define the workflow edges
workflow.add_edge(START, "document_verification")

workflow.add_edge("document_verification", END)

# Compile the graph
document_graph = workflow.compile()

print("LangGraph workflow compiled successfully.")
print("Current workflow: START → Document Verification → END")

LangGraph workflow compiled successfully.
Current workflow: START → Document Verification → END


## 9. Execute the LangGraph Workflow

Invoke the compiled LangGraph workflow using the initial claim state.

LangGraph will:

1. Start with the initial claim state.
2. Execute the Document Verification Agent.
3. Update the shared state with the verification result.
4. Return the updated state.

This is our first complete execution of a LangGraph workflow.

In [15]:
result_state = document_graph.invoke(initial_state)

print("LangGraph execution completed successfully.")
print("\nDocument Status:", result_state["document_status"])
print("Missing Documents:", result_state["missing_documents"])

LangGraph execution completed successfully.

Document Status: Complete
Missing Documents: []


## 10. Eligibility Check Agent

The Eligibility Check Agent determines whether the claim is eligible under the insurance policy.

The agent will evaluate:

- Whether the policy is currently active.
- Whether the policy was valid on the claim date.
- Whether the claim date is before the policy expiry date.
- Whether the claim has a valid claim amount.

For this project, the eligibility decision will be based on explicit business rules rather than unrestricted LLM reasoning.

Possible outcomes:

- **Eligible** → The policy is valid and the claim satisfies the basic eligibility rules.
- **Not Eligible** → The policy is inactive, expired, or otherwise fails the eligibility checks.

In [ ]:
def eligibility_check_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    claim = state["claim"]

    if claim.policy_status.strip().lower() != "active":
        return {
            "eligibility_status": "Not Eligible",
            "eligibility_reason": "The insurance policy is not active."
        }

    if claim.claim_date > claim.policy_expiry_date:
        return {
            "eligibility_status": "Not Eligible",
            "eligibility_reason": "The claim was submitted after the policy expiry date."
        }

    if claim.claim_amount <= 0:
        return {
            "eligibility_status": "Not Eligible",
            "eligibility_reason": "The claim amount must be greater than zero."
        }

    return {
        "eligibility_status": "Eligible",
        "eligibility_reason": "The policy is active and the claim satisfies the basic eligibility checks."
    }


eligibility_result = eligibility_check_agent(initial_state)

print("Eligibility Check Result")
print("-----------------------")
print("Status:", eligibility_result["eligibility_status"])
print("Reason:", eligibility_result["eligibility_reason"])

Eligibility Check Result
-----------------------
Status: Eligible
Reason: The policy is active and the claim satisfies the basic eligibility checks.


## 11. Fraud Detection Agent

The Fraud Detection Agent identifies potential fraud indicators in an insurance claim.

The agent will evaluate factors such as:

- Unusually high claim amounts
- Suspicious or inconsistent claim information
- Other predefined risk indicators

For this initial implementation, the claim amount will be used as a simple business rule:

- **Low Risk** → Claim amount below $10,000
- **Medium Risk** → Claim amount from $10,000 to $25,000
- **High Risk** → Claim amount above $25,000

The LLM will then provide a structured fraud-risk assessment based on the claim information and these rules.

This approach combines deterministic business rules with LLM-based reasoning and makes the fraud assessment easier to audit.

In [ ]:
from pydantic import BaseModel, Field

class FraudAssessment(BaseModel):
    risk_level: str = Field(description="Fraud risk level: Low, Medium, or High")
    indicators: list[str] = Field(description="Potential fraud indicators identified in the claim")

def fraud_detection_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    claim = state["claim"]

    if claim.claim_amount > 25000:
        amount_risk = "High"
    elif claim.claim_amount >= 10000:
        amount_risk = "Medium"
    else:
        amount_risk = "Low"

    structured_llm = llm.with_structured_output(FraudAssessment)

    prompt = f"""
You are an insurance fraud risk assessment assistant.

Assess the following insurance claim for potential fraud indicators.

Claim ID: {claim.claim_id}
Policy ID: {claim.policy_id}
Claim Type: {claim.claim_type}
Claim Amount: ${claim.claim_amount:,.2f}
Policy Status: {claim.policy_status}
Document Status: {state.get("document_status", "Not assessed")}
Amount-Based Risk: {amount_risk}

Use the following risk thresholds:
- Below $10,000: Low
- $10,000 to $25,000: Medium
- Above $25,000: High

Return:
1. A risk level of Low, Medium, or High.
2. A list of specific indicators supporting the assessment.

Do not invent facts that are not present in the claim.
"""

    assessment = structured_llm.invoke(prompt)

    return {
        "fraud_risk_level": assessment.risk_level,
        "fraud_indicators": assessment.indicators
    }


fraud_result = fraud_detection_agent(initial_state)

print("Fraud Detection Result")
print("----------------------")
print("Risk Level:", fraud_result["fraud_risk_level"])
print("Indicators:", fraud_result["fraud_indicators"])

Fraud Detection Result
----------------------
Risk Level: Low
Indicators: ['Claim amount is below $10,000, which is categorized as low risk.', 'Policy status is active, indicating coverage is valid.', 'Amount-based risk assessment is low.']


## 12. Parallel Assessment Workflow

The Document Verification, Eligibility Check, and Fraud Detection Agents are independent of one another.

LangGraph can therefore execute these three nodes in parallel from the same starting point.

This design:

- Reduces unnecessary sequential processing.
- Demonstrates parallel node execution.
- Allows each specialized agent to independently assess the claim.
- Combines their results into a shared state for downstream decision-making.

### Workflow

START
  │
  ├── Document Verification
  │
  ├── Eligibility Check
  │
  └── Fraud Detection
          │
          ▼
    Assessment Results
          │
         END

The three agents will later feed into a decision-routing node.

In [18]:
# Create a new LangGraph workflow for parallel claim assessment
parallel_workflow = StateGraph(ClaimProcessingState)

# Add the three assessment agents as nodes
parallel_workflow.add_node("document_verification", document_verification_agent)
parallel_workflow.add_node("eligibility_check", eligibility_check_agent)
parallel_workflow.add_node("fraud_detection", fraud_detection_agent)

# Start all three assessment agents from START
parallel_workflow.add_edge(START, "document_verification")
parallel_workflow.add_edge(START, "eligibility_check")
parallel_workflow.add_edge(START, "fraud_detection")

# End the workflow after all three parallel branches complete
parallel_workflow.add_edge("document_verification", END)
parallel_workflow.add_edge("eligibility_check", END)
parallel_workflow.add_edge("fraud_detection", END)

# Compile the graph
parallel_graph = parallel_workflow.compile()

print("Parallel LangGraph workflow compiled successfully.")
print("Assessment agents: Document Verification, Eligibility Check, Fraud Detection")

Parallel LangGraph workflow compiled successfully.
Assessment agents: Document Verification, Eligibility Check, Fraud Detection


## 13. Execute the Parallel Assessment Workflow

Invoke the parallel LangGraph workflow using the initial claim state.

The workflow will execute:

- Document Verification Agent
- Eligibility Check Agent
- Fraud Detection Agent

The results from all three agents will be merged into the shared LangGraph state.

This confirms that the core parallel assessment stage of our AgenticAI workflow is functioning correctly.

In [19]:
parallel_result = parallel_graph.invoke(initial_state)

print("Parallel assessment completed successfully.")
print("\nAssessment Results")
print("------------------")
print("Document Status:", parallel_result["document_status"])
print("Missing Documents:", parallel_result["missing_documents"])
print("Eligibility Status:", parallel_result["eligibility_status"])
print("Eligibility Reason:", parallel_result["eligibility_reason"])
print("Fraud Risk Level:", parallel_result["fraud_risk_level"])
print("Fraud Indicators:", parallel_result["fraud_indicators"])

Parallel assessment completed successfully.

Assessment Results
------------------
Document Status: Complete
Missing Documents: []
Eligibility Status: Eligible
Eligibility Reason: The policy is active and the claim satisfies the basic eligibility checks.
Fraud Risk Level: Low
Fraud Indicators: ['Claim amount is below $10,000, which is categorized as low risk.', 'Policy status is active, indicating that the policy is valid and current.']


## 14. Decision Router

The Decision Router evaluates the results produced by the three assessment agents and determines the next step in the claim-processing workflow.

### Routing Rules

**Reject** when:

- Required documents are missing, or
- The policy is not eligible.

**Human Review** when:

- Fraud risk is High, or
- The claim amount is greater than $25,000.

**Continue to Claim Summary** when:

- Documents are complete.
- The policy is eligible.
- Fraud risk is Low or Medium.
- The claim does not meet the human-review criteria.

The router uses deterministic business rules so that critical claim-routing decisions are predictable, auditable, and consistent.

In [ ]:
def decision_router(state: ClaimProcessingState) -> str:
    claim = state["claim"]

    # Reject if required documents are missing
    if state["document_status"] != "Complete":
        return "reject"

    # Reject if the policy is not eligible
    if state["eligibility_status"] != "Eligible":
        return "reject"

    # Escalate high-risk claims for human review
    if state["fraud_risk_level"] == "High":
        return "human_review"

    # Escalate high-value claims for human review
    if claim.claim_amount > 25000:
        return "human_review"

    # Continue with claim summary for low/medium-risk eligible claims
    return "claim_summary"


routing_decision = decision_router(parallel_result)

print("Decision Router Result")
print("----------------------")
print("Next Step:", routing_decision)

Decision Router Result
----------------------
Next Step: claim_summary


## 15. Add Conditional Edges to LangGraph

Connect the three parallel assessment agents to a Decision Router and use LangGraph conditional edges to determine the next step.

The Decision Router will return one of three route names:

- `reject`
- `human_review`
- `claim_summary`

LangGraph will use these values to follow the appropriate conditional edge.

For this stage, the Reject and Human Review branches will end the workflow temporarily.

The Claim Summary branch will be connected to the Claim Summary Agent in the next step.

In [21]:
# Create the complete assessment and routing workflow
routing_workflow = StateGraph(ClaimProcessingState)

# Add assessment nodes
routing_workflow.add_node("document_verification", document_verification_agent)
routing_workflow.add_node("eligibility_check", eligibility_check_agent)
routing_workflow.add_node("fraud_detection", fraud_detection_agent)

# Add the Decision Router node
routing_workflow.add_node("decision_router", lambda state: state)

# Parallel execution from START
routing_workflow.add_edge(START, "document_verification")
routing_workflow.add_edge(START, "eligibility_check")
routing_workflow.add_edge(START, "fraud_detection")

# All three assessments feed into the Decision Router
routing_workflow.add_edge("document_verification", "decision_router")
routing_workflow.add_edge("eligibility_check", "decision_router")
routing_workflow.add_edge("fraud_detection", "decision_router")

# Add conditional routing
routing_workflow.add_conditional_edges("decision_router", decision_router,
    {
        "reject": END,
        "human_review": END,
        "claim_summary": END
    }
)

# Compile the graph
routing_graph = routing_workflow.compile()

print("Conditional routing workflow compiled successfully.")
print("Routes: Reject | Human Review | Claim Summary")

Conditional routing workflow compiled successfully.
Routes: Reject | Human Review | Claim Summary


## 16. Claim Summary Agent

The Claim Summary Agent generates a concise, structured summary of the insurance claim using the results produced by the assessment agents.

The summary will include:

- Claim and policy information
- Document verification status
- Eligibility result
- Fraud risk level
- Fraud indicators
- Claim amount
- Recommended next action

The LLM will be instructed to use only the information available in the claim-processing state and not to invent facts.

This summary will be used as part of the final claim-processing result and will also provide useful context if the claim is later escalated for human review.

In [ ]:
def claim_summary_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    claim = state["claim"]

    prompt = f"""
You are an insurance claim processing assistant.

Generate a concise professional summary of the following insurance claim.

Claim Details:
- Claim ID: {claim.claim_id}
- Policy ID: {claim.policy_id}
- Claimant: {claim.claimant_name}
- Claim Type: {claim.claim_type}
- Claim Amount: ${claim.claim_amount:,.2f}
- Policy Status: {claim.policy_status}
- Policy Expiry Date: {claim.policy_expiry_date}
- Claim Date: {claim.claim_date}

Assessment Results:
- Document Status: {state.get("document_status", "Not assessed")}
- Missing Documents: {state.get("missing_documents", [])}
- Eligibility Status: {state.get("eligibility_status", "Not assessed")}
- Eligibility Reason: {state.get("eligibility_reason", "Not available")}
- Fraud Risk Level: {state.get("fraud_risk_level", "Not assessed")}
- Fraud Indicators: {state.get("fraud_indicators", [])}

Write a concise claim summary covering the key facts and assessment results.

Do not invent information that is not provided above.
"""

    response = llm.invoke(prompt)
    return {"claim_summary": response.content}

summary_result = claim_summary_agent(parallel_result)

print("Claim Summary Generated Successfully")
print("-------------------------------------")
print(summary_result["claim_summary"])

Claim Summary Generated Successfully
-------------------------------------
**Claim Summary:**

- **Claim ID:** CLM-1001  
- **Policy ID:** POL-5001  
- **Claimant:** John Smith  
- **Claim Type:** Vehicle Damage  
- **Claim Amount:** $8,500.00  
- **Policy Status:** Active (Expiry Date: 2027-12-31)  
- **Claim Date:** 2026-09-15  

**Assessment Results:**  
- **Document Status:** Complete  
- **Missing Documents:** None  
- **Eligibility Status:** Eligible  
- **Eligibility Reason:** The policy is active and the claim meets basic eligibility checks.  
- **Fraud Risk Level:** Low  
- **Fraud Indicators:** Claim amount is below $10,000 and the policy is valid and current.  

This claim is eligible for processing based on the active policy and completed documentation.


## 17. Connect the Claim Summary Agent

Add the Claim Summary Agent to the LangGraph workflow.

For an eligible, low-risk claim that does not require human review:

**Parallel Assessment → Decision Router → Claim Summary → Auto Approve**

The Claim Summary Agent will generate the final claim summary, after which the workflow will assign the final decision as **Auto Approved**.

The Reject and Human Review routes will remain available for the next stages of development.

In [ ]:
def auto_approve_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    return {
        "final_decision": "Auto Approved",
        "human_review_required": False
    }

# Create the workflow
claim_workflow = StateGraph(ClaimProcessingState)

# Add assessment nodes
claim_workflow.add_node("document_verification", document_verification_agent)
claim_workflow.add_node("eligibility_check", eligibility_check_agent)
claim_workflow.add_node("fraud_detection", fraud_detection_agent)

# Add routing node
claim_workflow.add_node("decision_router", lambda state: state)

# Add Claim Summary Agent
claim_workflow.add_node("claim_summary", claim_summary_agent)

# Add Auto Approval node
claim_workflow.add_node("auto_approve", auto_approve_agent)

# Start parallel assessment
claim_workflow.add_edge(START, "document_verification")
claim_workflow.add_edge(START, "eligibility_check")
claim_workflow.add_edge(START, "fraud_detection")

# Assessment nodes feed into the Decision Router
claim_workflow.add_edge("document_verification", "decision_router")
claim_workflow.add_edge("eligibility_check", "decision_router")
claim_workflow.add_edge("fraud_detection", "decision_router")

# Conditional routing
claim_workflow.add_conditional_edges("decision_router", decision_router,
    {
        "reject": END,
        "human_review": END,
        "claim_summary": "claim_summary"
    }
)

# Successful claims move from summary to auto approval
claim_workflow.add_edge("claim_summary", "auto_approve")
claim_workflow.add_edge("auto_approve", END)

# Compile the graph
claim_graph = claim_workflow.compile()

print("Complete claim workflow compiled successfully.")
print("Workflow includes parallel assessment, conditional routing, claim summary, and automatic approval.")

Complete claim workflow compiled successfully.
Workflow includes parallel assessment, conditional routing, claim summary, and automatic approval.


## 18. End-to-End Test — Auto-Approve Scenario

Run the complete LangGraph workflow using the valid sample claim.

Expected workflow:

**Claim Input**
→ **Parallel Assessment**
→ **Decision Router**
→ **Claim Summary**
→ **Auto Approve**

Expected outcome:

- Documents: Complete
- Eligibility: Eligible
- Fraud Risk: Low
- Final Decision: Auto Approved
- Human Review: Not Required

In [25]:
final_result = claim_graph.invoke(initial_state)

print("END-TO-END CLAIM PROCESSING RESULT")
print("===================================")
print("Claim ID:", final_result["claim"].claim_id)
print("Document Status:", final_result["document_status"])
print("Eligibility Status:", final_result["eligibility_status"])
print("Fraud Risk Level:", final_result["fraud_risk_level"])
print("Human Review Required:", final_result["human_review_required"])
print("Final Decision:", final_result["final_decision"])

print("\nClaim Summary")
print("-------------")
print(final_result["claim_summary"])

END-TO-END CLAIM PROCESSING RESULT
Claim ID: CLM-1001
Document Status: Complete
Eligibility Status: Eligible
Fraud Risk Level: Low
Human Review Required: False
Final Decision: Auto Approved

Claim Summary
-------------
**Claim Summary:**

- **Claim ID:** CLM-1001  
- **Policy ID:** POL-5001  
- **Claimant:** John Smith  
- **Claim Type:** Vehicle Damage  
- **Claim Amount:** $8,500.00  
- **Policy Status:** Active (Expiry Date: 2027-12-31)  
- **Claim Date:** 2026-09-15  

**Assessment Results:**  
- **Document Status:** Complete  
- **Missing Documents:** None  
- **Eligibility Status:** Eligible  
- **Eligibility Reason:** The policy is active and the claim meets basic eligibility checks.  
- **Fraud Risk Level:** Low  
- **Fraud Indicators:** Claim amount is below $10,000, policy status is active, and amount-based risk assessment is low.  

This claim is eligible for processing based on the active policy and completed documentation.


## 19. Human-in-the-Loop — Human Approval Agent

Some insurance claims should not be automatically approved.

Examples include:
- High fraud risk
- High-value claims
- Claims requiring additional investigation

For these cases, the workflow will pause and request a human decision.

The Human Approval Agent will:
1. Display the claim and assessment information.
2. Explain why human review is required.
3. Pause the LangGraph workflow.
4. Wait for a human decision.
5. Resume the workflow with the human's decision.

This implements Human-in-the-Loop (HITL) using LangGraph's `interrupt()` mechanism.

In [26]:
from langgraph.types import interrupt

def human_approval_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    claim = state["claim"]

    review_reason = []

    if state.get("fraud_risk_level") == "High":
        review_reason.append("High fraud risk detected.")

    if claim.claim_amount > 25000:
        review_reason.append("Claim amount exceeds the automatic approval threshold.")

    if not review_reason:
        review_reason.append("Claim requires manual review based on workflow rules.")

    human_review_reason = " ".join(review_reason)

    approval_request = {
        "message": "Human approval is required for this claim.",
        "claim_id": claim.claim_id,
        "claimant_name": claim.claimant_name,
        "claim_amount": claim.claim_amount,
        "fraud_risk_level": state.get("fraud_risk_level"),
        "fraud_indicators": state.get("fraud_indicators", []),
        "eligibility_status": state.get("eligibility_status"),
        "document_status": state.get("document_status"),
        "claim_summary": state.get("claim_summary", ""),
        "review_reason": human_review_reason
    }

    human_decision = interrupt(approval_request)

    return {
        "final_decision": human_decision,
        "human_review_required": True,
        "human_review_reason": human_review_reason
    }

print("Human Approval Agent created successfully.")

Human Approval Agent created successfully.


## 20. Updated Workflow with Human Approval

The workflow will now support three possible outcomes:

1. **Auto Approve**
   - Complete documents
   - Eligible policy
   - Low/Medium fraud risk
   - Claim amount within automatic approval limit

2. **Reject**
   - Missing required documents
   - Ineligible or expired policy

3. **Human Review**
   - High fraud risk
   - High-value claim
   - Claim summary is generated before requesting human approval

Updated workflow:

**Claim Input**
→ **Parallel Assessment**
→ **Decision Router**

**Reject**
→ **Reject Claim**

**Human Review**
→ **Claim Summary**
→ **Human Approval**
→ **END**

**Auto Approve**
→ **Claim Summary**
→ **Auto Approve**
→ **END**

In [27]:
def reject_claim_agent(state: ClaimProcessingState) -> ClaimProcessingState:
    if state.get("document_status") != "Complete":
        reason = (
            f"Required documents are missing: "
            f"{', '.join(state.get('missing_documents', []))}."
        )
    elif state.get("eligibility_status") != "Eligible":
        reason = state.get(
            "eligibility_reason",
            "The claim does not meet the policy eligibility requirements."
        )
    else:
        reason = "The claim does not meet the automatic processing requirements."

    return {
        "final_decision": "Rejected",
        "human_review_required": False,
        "human_review_reason": reason
    }


def decision_router_v2(state: ClaimProcessingState) -> str:
    claim = state["claim"]

    if state["document_status"] != "Complete":
        return "reject"

    if state["eligibility_status"] != "Eligible":
        return "reject"

    if state["fraud_risk_level"] == "High":
        return "human_review"

    if claim.claim_amount > 25000:
        return "human_review"

    return "auto_approve"

updated_workflow = StateGraph(ClaimProcessingState)

updated_workflow.add_node("document_verification", document_verification_agent)
updated_workflow.add_node("eligibility_check", eligibility_check_agent)
updated_workflow.add_node("fraud_detection", fraud_detection_agent)
updated_workflow.add_node("decision_router", lambda state: state)
updated_workflow.add_node("reject_claim", reject_claim_agent)
updated_workflow.add_node("claim_summary", claim_summary_agent)
updated_workflow.add_node("human_approval", human_approval_agent)
updated_workflow.add_node("auto_approve", auto_approve_agent)

updated_workflow.add_edge(START, "document_verification")
updated_workflow.add_edge(START, "eligibility_check")
updated_workflow.add_edge(START, "fraud_detection")

updated_workflow.add_edge("document_verification", "decision_router")
updated_workflow.add_edge("eligibility_check", "decision_router")
updated_workflow.add_edge("fraud_detection", "decision_router")

updated_workflow.add_conditional_edges(
    "decision_router",
    decision_router_v2,
    {
        "reject": "reject_claim",
        "human_review": "claim_summary",
        "auto_approve": "claim_summary"
    }
)

updated_workflow.add_edge("reject_claim", END)

updated_workflow.add_conditional_edges(
    "claim_summary",
    lambda state: (
        "human_approval"
        if decision_router_v2(state) == "human_review"
        else "auto_approve"
    ),
    {
        "human_approval": "human_approval",
        "auto_approve": "auto_approve"
    }
)

updated_workflow.add_edge("auto_approve", END)
updated_workflow.add_edge("human_approval", END)

print("Updated Human-in-the-Loop workflow compiled successfully.")
print()
print("Workflow supports:")
print("- Automatic Approval")
print("- Rejection")
print("- Human Review")

Updated Human-in-the-Loop workflow compiled successfully.

Workflow supports:
- Automatic Approval
- Rejection
- Human Review


## 21. Enable Checkpointing for Human-in-the-Loop

LangGraph needs to preserve the workflow state when `interrupt()` pauses execution.

A checkpointer allows us to:
- Pause the workflow.
- Preserve the current claim state.
- Wait for human input.
- Resume the same workflow later.

For this project, we will use `MemorySaver`, which stores checkpoints in memory.

This is suitable for our development and demonstration environment.
For a production application, a persistent database-backed checkpointer would be preferred.

In [28]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
claim_graph_hitl = updated_workflow.compile(checkpointer=memory)

print("HITL workflow compiled with checkpointing successfully.")
print("Checkpointing enabled:", True)

HITL workflow compiled with checkpointing successfully.
Checkpointing enabled: True


## 22. Human Review Test — High-Value Claim

We will test a valid claim with a high claim amount.

Expected outcome:

- Documents: Complete
- Eligibility: Eligible
- Claim Amount: $30,000
- Fraud Risk: High
- Human Review: Required
- Workflow: Pauses for human approval

The workflow should stop at the Human Approval Agent and wait for a human decision.

In [29]:
high_value_claim = InsuranceClaim(
    claim_id="CLM-1002",
    policy_id="POL-5002",
    claimant_name="Sarah Johnson",
    policy_status="Active",
    policy_expiry_date=date(2027, 12, 31),
    claim_date=date(2026, 9, 20),
    claim_type="Vehicle Damage",
    claim_amount=30000.00,
    required_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ],
    submitted_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ]
)

high_value_state: ClaimProcessingState = {
    "claim": high_value_claim,
    "document_status": "",
    "missing_documents": [],
    "eligibility_status": "",
    "eligibility_reason": "",
    "fraud_risk_level": "",
    "fraud_indicators": [],
    "claim_summary": "",
    "final_decision": "",
    "human_review_required": False,
    "human_review_reason": ""
}

print("High-value claim created successfully.")
print()
print("Claim ID:", high_value_claim.claim_id)
print("Claim Amount:", f"${high_value_claim.claim_amount:,.2f}")
print("Policy Status:", high_value_claim.policy_status)

High-value claim created successfully.

Claim ID: CLM-1002
Claim Amount: $30,000.00
Policy Status: Active


## 23. Execute Human-in-the-Loop Workflow

Run the high-value claim through the complete workflow.

The workflow should:
1. Verify the documents.
2. Check policy eligibility.
3. Assess fraud risk.
4. Generate the claim summary.
5. Route the claim to Human Approval.
6. Pause using `interrupt()`.

At this point, we will NOT provide a human decision yet.

The purpose of this test is to confirm that the workflow can successfully pause and wait for human intervention.

In [30]:
from langgraph.types import Command

thread_config = {"configurable": {"thread_id": high_value_claim.claim_id}}

hitl_result = claim_graph_hitl.invoke(high_value_state, config=thread_config)

print("HITL workflow execution paused successfully.")
print()
print("Claim ID:", high_value_claim.claim_id)
print("Workflow is waiting for human approval.")
print()
print("Interrupt Information:")
print(hitl_result)

HITL workflow execution paused successfully.

Claim ID: CLM-1002
Workflow is waiting for human approval.

Interrupt Information:
{'claim': InsuranceClaim(claim_id='CLM-1002', policy_id='POL-5002', claimant_name='Sarah Johnson', policy_status='Active', policy_expiry_date=datetime.date(2027, 12, 31), claim_date=datetime.date(2026, 9, 20), claim_type='Vehicle Damage', claim_amount=30000.0, required_documents=['Claim Form', 'Policy Document', 'Identity Proof', 'Repair Estimate'], submitted_documents=['Claim Form', 'Policy Document', 'Identity Proof', 'Repair Estimate']), 'document_status': 'Complete', 'missing_documents': [], 'eligibility_status': 'Eligible', 'eligibility_reason': 'The policy is active and the claim satisfies the basic eligibility checks.', 'fraud_risk_level': 'High', 'fraud_indicators': ['Claim amount of $30,000 exceeds the high-risk threshold for fraud assessment.', 'Amount-based risk is categorized as High, indicating potential for inflated claims.', 'The claim type is 

## 24. Resume Workflow After Human Approval

The workflow is currently paused and waiting for a human decision.

We will now simulate a human reviewer approving the claim.

The human decision will be passed back to LangGraph using `Command(resume=...)`.

Expected outcome:

- Human Review: Completed
- Human Decision: Approved
- Final Decision: Approved by Human
- Workflow: Completed

In [31]:
human_decision = "Approved by Human"

final_hitl_result = claim_graph_hitl.invoke(Command(resume=human_decision), config=thread_config)

print("Human-in-the-Loop workflow completed successfully.")
print()
print("Claim ID:", final_hitl_result["claim"].claim_id)
print("Fraud Risk Level:", final_hitl_result["fraud_risk_level"])
print("Human Review Required:", final_hitl_result["human_review_required"])
print("Human Review Reason:", final_hitl_result["human_review_reason"])
print("Final Decision:", final_hitl_result["final_decision"])

Deserializing unregistered type __main__.InsuranceClaim from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'InsuranceClaim')]


Human-in-the-Loop workflow completed successfully.

Claim ID: CLM-1002
Fraud Risk Level: High
Human Review Required: True
Human Review Reason: High fraud risk detected. Claim amount exceeds the automatic approval threshold.
Final Decision: Approved by Human


## 25. Reject Scenario — Missing Required Documents

We will now test a claim where required documents are missing.

Expected outcome:

- Documents: Incomplete
- Missing Documents: Identity Proof and Repair Estimate
- Eligibility: Eligible
- Fraud Risk: Low
- Final Decision: Rejected
- Human Review: Not Required

Because required documents are missing, the Decision Router should send the claim directly to the Reject Claim Agent.

In [32]:
missing_documents_claim = InsuranceClaim(
    claim_id="CLM-1003",
    policy_id="POL-5003",
    claimant_name="Michael Brown",
    policy_status="Active",
    policy_expiry_date=date(2027, 12, 31),
    claim_date=date(2026, 9, 25),
    claim_type="Vehicle Damage",
    claim_amount=7500.00,
    required_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ],
    submitted_documents=[
        "Claim Form",
        "Policy Document"
    ]
)

missing_documents_state: ClaimProcessingState = {
    "claim": missing_documents_claim,
    "document_status": "",
    "missing_documents": [],
    "eligibility_status": "",
    "eligibility_reason": "",
    "fraud_risk_level": "",
    "fraud_indicators": [],
    "claim_summary": "",
    "final_decision": "",
    "human_review_required": False,
    "human_review_reason": ""
}

print("Missing-document claim created successfully.")
print()
print("Claim ID:", missing_documents_claim.claim_id)
print("Claim Amount:", f"${missing_documents_claim.claim_amount:,.2f}")
print("Submitted Documents:", missing_documents_claim.submitted_documents)

Missing-document claim created successfully.

Claim ID: CLM-1003
Claim Amount: $7,500.00
Submitted Documents: ['Claim Form', 'Policy Document']


## 26. Execute Reject Scenario

Run the incomplete claim through the complete workflow.

Expected outcome:

- Document Status: Incomplete
- Missing Documents: Identity Proof, Repair Estimate
- Eligibility Status: Eligible
- Fraud Risk: Low
- Final Decision: Rejected
- Human Review: Not Required

The claim should be rejected by the Reject Claim Agent because required documents are missing.

In [33]:
reject_thread_config = {"configurable": {"thread_id": missing_documents_claim.claim_id}}

reject_result = claim_graph_hitl.invoke(missing_documents_state, config=reject_thread_config)

print("Reject scenario completed successfully.")
print()
print("Claim ID:", reject_result["claim"].claim_id)
print("Document Status:", reject_result["document_status"])
print("Missing Documents:", reject_result["missing_documents"])
print("Eligibility Status:", reject_result["eligibility_status"])
print("Fraud Risk Level:", reject_result["fraud_risk_level"])
print("Human Review Required:", reject_result["human_review_required"])
print("Final Decision:", reject_result["final_decision"])
print("Rejection/Review Reason:", reject_result["human_review_reason"])

Reject scenario completed successfully.

Claim ID: CLM-1003
Document Status: Incomplete
Missing Documents: ['identity proof', 'repair estimate']
Eligibility Status: Eligible
Fraud Risk Level: Low
Human Review Required: False
Final Decision: Rejected
Rejection/Review Reason: Required documents are missing: identity proof, repair estimate.


## 27. Reject Scenario — Expired Policy

We will now test a claim submitted after the insurance policy has expired.

Expected outcome:

- Documents: Complete
- Eligibility: Not Eligible
- Eligibility Reason: Claim submitted after policy expiry
- Fraud Risk: Low
- Final Decision: Rejected
- Human Review: Not Required

The Decision Router should reject the claim because the policy is no longer valid for the claim date.

In [34]:
expired_policy_claim = InsuranceClaim(
    claim_id="CLM-1004",
    policy_id="POL-5004",
    claimant_name="David Wilson",
    policy_status="Active",
    policy_expiry_date=date(2026, 8, 31),
    claim_date=date(2026, 9, 15),
    claim_type="Vehicle Damage",
    claim_amount=9000.00,
    required_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ],
    submitted_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Repair Estimate"
    ]
)

expired_policy_state: ClaimProcessingState = {
    "claim": expired_policy_claim,
    "document_status": "",
    "missing_documents": [],
    "eligibility_status": "",
    "eligibility_reason": "",
    "fraud_risk_level": "",
    "fraud_indicators": [],
    "claim_summary": "",
    "final_decision": "",
    "human_review_required": False,
    "human_review_reason": ""
}

print("Expired-policy claim created successfully.")
print()
print("Claim ID:", expired_policy_claim.claim_id)
print("Policy Expiry Date:", expired_policy_claim.policy_expiry_date)
print("Claim Date:", expired_policy_claim.claim_date)
print("Submitted Documents:", expired_policy_claim.submitted_documents)

Expired-policy claim created successfully.

Claim ID: CLM-1004
Policy Expiry Date: 2026-08-31
Claim Date: 2026-09-15
Submitted Documents: ['Claim Form', 'Policy Document', 'Identity Proof', 'Repair Estimate']


## 28. Execute Expired Policy Scenario

Run the expired-policy claim through the complete workflow.

Expected outcome:

- Document Status: Complete
- Eligibility Status: Not Eligible
- Eligibility Reason: Claim submitted after policy expiry
- Fraud Risk: Low
- Final Decision: Rejected
- Human Review: Not Required

In [35]:
expired_thread_config = {"configurable": {"thread_id": expired_policy_claim.claim_id}}

expired_result = claim_graph_hitl.invoke(expired_policy_state, config=expired_thread_config)

print("Expired-policy scenario completed successfully.")
print()
print("Claim ID:", expired_result["claim"].claim_id)
print("Document Status:", expired_result["document_status"])
print("Missing Documents:", expired_result["missing_documents"])
print("Eligibility Status:", expired_result["eligibility_status"])
print("Eligibility Reason:", expired_result["eligibility_reason"])
print("Fraud Risk Level:", expired_result["fraud_risk_level"])
print("Human Review Required:", expired_result["human_review_required"])
print("Final Decision:", expired_result["final_decision"])
print("Rejection/Review Reason:", expired_result["human_review_reason"])

Expired-policy scenario completed successfully.

Claim ID: CLM-1004
Document Status: Complete
Missing Documents: []
Eligibility Status: Not Eligible
Eligibility Reason: The claim was submitted after the policy expiry date.
Fraud Risk Level: Low
Human Review Required: False
Final Decision: Rejected
Rejection/Review Reason: The claim was submitted after the policy expiry date.


## 29. Human Review Scenario — High-Value Claim

We will test a high-value claim that is otherwise valid.

Expected outcome:

- Documents: Complete
- Eligibility: Eligible
- Claim Amount: $26,000
- Fraud Risk: High or Medium based on the assessment
- Human Review: Required
- Final Decision: Human Review

The claim should be routed to Human Approval because the claim amount exceeds the $25,000 automatic approval threshold.

In [36]:
high_value_valid_claim = InsuranceClaim(
    claim_id="CLM-1005",
    policy_id="POL-5005",
    claimant_name="Robert Davis",
    policy_status="Active",
    policy_expiry_date=date(2027, 12, 31),
    claim_date=date(2026, 9, 28),
    claim_type="Property Damage",
    claim_amount=26000.00,
    required_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Damage Assessment"
    ],
    submitted_documents=[
        "Claim Form",
        "Policy Document",
        "Identity Proof",
        "Damage Assessment"
    ]
)

high_value_valid_state: ClaimProcessingState = {
    "claim": high_value_valid_claim,
    "document_status": "",
    "missing_documents": [],
    "eligibility_status": "",
    "eligibility_reason": "",
    "fraud_risk_level": "",
    "fraud_indicators": [],
    "claim_summary": "",
    "final_decision": "",
    "human_review_required": False,
    "human_review_reason": ""
}

print("High-value valid claim created successfully.")
print()
print("Claim ID:", high_value_valid_claim.claim_id)
print("Claim Type:", high_value_valid_claim.claim_type)
print("Claim Amount:", f"${high_value_valid_claim.claim_amount:,.2f}")
print("Policy Status:", high_value_valid_claim.policy_status)

High-value valid claim created successfully.

Claim ID: CLM-1005
Claim Type: Property Damage
Claim Amount: $26,000.00
Policy Status: Active


## 30. Execute High-Value Claim Scenario

Run the high-value claim through the complete workflow.

The claim has:
- Complete documentation
- Active policy
- Claim amount above $25,000

Expected outcome:

**Parallel Assessment → Human Review → Claim Summary → Human Approval**

The workflow should pause at the Human Approval Agent.

In [37]:
high_value_thread_config = {"configurable": {"thread_id": high_value_valid_claim.claim_id}}

high_value_result = claim_graph_hitl.invoke(high_value_valid_state, config=high_value_thread_config)

print("High-value claim workflow paused for human review.")
print()
print("Claim ID:", high_value_result["claim"].claim_id)
print("Document Status:", high_value_result["document_status"])
print("Eligibility Status:", high_value_result["eligibility_status"])
print("Fraud Risk Level:", high_value_result["fraud_risk_level"])
print("Human Review Required:", "Yes" if "__interrupt__" in high_value_result else "No")

if "__interrupt__" in high_value_result:
    print("Workflow Status: Waiting for Human Approval")
else:
    print("Final Decision:", high_value_result.get("final_decision"))

High-value claim workflow paused for human review.

Claim ID: CLM-1005
Document Status: Complete
Eligibility Status: Eligible
Fraud Risk Level: High
Human Review Required: Yes
Workflow Status: Waiting for Human Approval


## 31. Resume High-Value Claim After Human Approval

The claim has been reviewed by a human.

For this test, the human reviewer approves the claim.

Expected outcome:

- Human Review: Completed
- Human Decision: Approved by Human
- Final Decision: Approved by Human
- Workflow: Completed

In [38]:
final_high_value_result = claim_graph_hitl.invoke(Command(resume="Approved by Human"), config=high_value_thread_config)

print("High-value claim human review completed successfully.")
print()
print("Claim ID:", final_high_value_result["claim"].claim_id)
print("Fraud Risk Level:", final_high_value_result["fraud_risk_level"])
print("Human Review Required:", final_high_value_result["human_review_required"])
print("Human Review Reason:", final_high_value_result["human_review_reason"])
print("Final Decision:", final_high_value_result["final_decision"])

High-value claim human review completed successfully.

Claim ID: CLM-1005
Fraud Risk Level: High
Human Review Required: True
Human Review Reason: High fraud risk detected. Claim amount exceeds the automatic approval threshold.
Final Decision: Approved by Human


## 32. Create a Reusable Claim Processing Function

The LangGraph workflow is currently executed directly using `claim_graph_hitl.invoke()`.

For the Streamlit application, we need a simple reusable function that accepts an insurance claim and starts the workflow.

This function will:
- Create the initial claim state.
- Create a unique workflow thread.
- Execute the LangGraph workflow.
- Return the processing result.

Keeping this logic separate from the Streamlit UI makes the application easier to maintain and test.

In [39]:
def create_claim_state(claim: InsuranceClaim) -> ClaimProcessingState:
    return {
        "claim": claim,
        "document_status": "",
        "missing_documents": [],
        "eligibility_status": "",
        "eligibility_reason": "",
        "fraud_risk_level": "",
        "fraud_indicators": [],
        "claim_summary": "",
        "final_decision": "",
        "human_review_required": False,
        "human_review_reason": ""
    }

def process_claim(claim: InsuranceClaim):
    claim_state = create_claim_state(claim)

    config = {"configurable": {"thread_id": claim.claim_id}}

    result = claim_graph_hitl.invoke(claim_state, config=config)

    return result, config

print("Reusable claim processing functions created successfully.")

Reusable claim processing functions created successfully.


## 33. Test the Reusable Claim Processing Function

Test the reusable `process_claim()` function using the original valid claim.

This confirms that the Streamlit application will be able to call the LangGraph workflow through a single function.

Expected outcome:

- Document Status: Complete
- Eligibility Status: Eligible
- Fraud Risk: Low
- Final Decision: Auto Approved
- No Human Review Required

In [40]:
processed_result, processed_config = process_claim(sample_claim)

print("Reusable claim processing function tested successfully.")
print()
print("Claim ID:", processed_result["claim"].claim_id)
print("Document Status:", processed_result["document_status"])
print("Eligibility Status:", processed_result["eligibility_status"])
print("Fraud Risk Level:", processed_result["fraud_risk_level"])
print("Human Review Required:", processed_result["human_review_required"])
print("Final Decision:", processed_result["final_decision"])

Reusable claim processing function tested successfully.

Claim ID: CLM-1001
Document Status: Complete
Eligibility Status: Eligible
Fraud Risk Level: Low
Human Review Required: False
Final Decision: Auto Approved


## 34. Prepare Supporting Document Handling

The Streamlit application will allow users to upload supporting documents.

For this assignment, document verification is based on whether the required document names are present among the submitted documents.

This helper function converts uploaded Streamlit files into a list of document names that can be passed to the LangGraph workflow.

In [41]:
def get_uploaded_document_names(uploaded_files) -> list[str]:
    if not uploaded_files:
        return []

    return [file.name for file in uploaded_files]

print("Document upload helper created successfully.")

Document upload helper created successfully.
